# Week 4: Unusual spending

Goal: find purchases (and weeks) that are far above **your usual**, so Ollie can say *"That ₱4,800 Shopping purchase is about 4× your usual."*

You'll compare three methods and choose the numbers the app ships: **multiplier**, **minimum amount**, **minimum earlier purchases**, **minimum earlier weeks**.

Spec: `docs/superpowers/specs/2026-10-22-ollie-anomaly-alerts-design.md`

## Setup
The data now has an `Anomaly` column: `spike` (one unusually big purchase), `spree` (many normal purchases in one week), or blank. These are the **answers**, so you can score each method.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support

df = pd.read_csv('../data/fake_transactions.csv')
df['Anomaly'] = df['Anomaly'].fillna('')

# Safety check: the generator only ADDED spree rows; every older row is unchanged
before = pd.read_csv('../data/fake_transactions_before.csv')
assert before.equals(df[df['Anomaly'] != 'spree'].drop(columns='Anomaly').reset_index(drop=True)), 'old rows changed!'

df['Date'] = pd.to_datetime(df['Date'])
exp = df[df['Type'] == 'expense'].sort_values('Date', kind='stable').reset_index(drop=True)
exp['Anomaly'].value_counts()

In [ ]:
def score(pred):
    """Precision, recall and F1 of a True/False prediction against the 'spike' labels."""
    p, r, f, _ = precision_recall_fscore_support(exp['Anomaly'] == 'spike', pred, average='binary', zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f}

---
## A. What is "usual"? (no looking ahead)
For each expense, **usual** = the median of **earlier** purchases in the same category. Using later purchases would be **leakage**: the app can't know the future.

Add three columns:
- `earlier`: how many earlier purchases in this category. Hint: `groupby('Category').cumcount()`
- `usual`: median of earlier amounts. Hint: `groupby('Category')['Amount'].transform(lambda s: s.shift().expanding().median())`
  - `shift()` drops the current row, so only earlier ones count
- `ratio`: `Amount / usual`

**Think:** why the median and not the mean? (One ₱20,000 spike would drag a mean up.)

In [ ]:
# TODO


These checks prove there's no look-ahead. If they pass, part A is right.

In [ ]:
# The first purchase in each category has nothing earlier
assert exp.groupby('Category').head(1)['usual'].isna().all()
# Recompute one row by hand from earlier rows only
row = exp[exp['earlier'] == 10].iloc[0]
hand = exp[(exp['Category'] == row['Category']) & (exp.index < row.name)]['Amount'].median()
assert abs(hand - row['usual']) < 1e-9
print('no look-ahead ✓')

---
## B. Three methods for single purchases
Score each with `score(...)`. It returns precision, recall and F1.

- **Precision:** of the alerts, how many were real spikes? (low = annoying false alarms)
- **Recall:** of the real spikes, how many did we catch?
- **F1:** one number balancing both

### B1. Ratio rule (what the app will use)
Flag when `ratio >= 3` **and** `Amount >= 500` **and** `earlier >= 5`. Save it as `ratio_pred`.

In [ ]:
# TODO


### B2. Robust z-score
How many *typical spreads* above usual is it?
- Spread = **MAD** (median absolute deviation) of earlier amounts. Hint: same `transform(lambda s: s.shift().expanding()...)` pattern, with `.apply(lambda w: np.nanmedian(np.abs(w - np.nanmedian(w))), raw=True)` (use `nanmedian`: `shift()` leaves a NaN that plain `np.median` turns into NaN for every row)
- `z = (Amount - usual) / (1.4826 * mad)`. Flag `z >= 3.5` and `earlier >= 5`. Save as `z_pred`.

In [ ]:
# TODO


### B3. Isolation Forest (real ML)
It learns what "normal" looks like and isolates rare points.
- Features: category one-hot (`pd.get_dummies(exp['Category']).astype(float)`) plus `np.log(exp['Amount'])`
- `IsolationForest(contamination=0.01, random_state=42)`, then `fit_predict(X) == -1` means anomaly. Save as `iso_pred`.

**Think:** Isolation Forest is fitted on *all* rows, future included. Is that a fair comparison with the ratio rule?

In [ ]:
# TODO


### B4. Compare
Put the three scores in one table. Hint: `pd.DataFrame({'ratio rule': score(ratio_pred), ...}).T.round(2)`

In [ ]:
# TODO


---
## C. Tune the ratio rule
Try multipliers `[2, 2.5, 3, 3.5, 4, 5]` × minimum amounts `[0, 250, 500, 1000]`. For each, record precision, recall, F1 and the number of alerts.

Hint: loop, build a list of dicts, `pd.DataFrame(rows)`, sort by `f1`.

**Think:** a lower multiplier catches more spikes. What does it cost the user?

In [ ]:
# TODO


---
## D. Unusual weeks (sprees)
Now the total per **category per week** (weeks start Monday).

1. Week label: `exp['Date'].dt.tz_localize(None).dt.to_period('W-SUN')`
2. Weekly totals: `pivot_table(index='week', columns='Category', values='Amount', aggfunc='sum', fill_value=0)`, then reindex to **every** week so empty weeks count as ₱0
3. Per category, starting at its first spending week: `usual` = median of earlier weeks (same `shift().expanding().median()` idea), `earlier_weeks` = count, `ratio` = total / usual
4. Build one table `weeks` with columns `Category, week, total, usual, earlier_weeks, ratio`

In [ ]:
# TODO


Score it: a (week, category) is a real spree if it contains any `spree` row. Flag `ratio >= 3`, `total >= 500`, `earlier_weeks >= 4`, `usual > 0`.

In [ ]:
# TODO


---
## E. Decide
**Gate:** if the ratio rule's F1 is more than **0.15** below Isolation Forest's, stop and tell Claude. We'd rethink the approach before building the app.

Otherwise, write your four numbers and one sentence for each saying why. These go into the app:
- `MULTIPLIER`
- `MIN_AMOUNT`
- `MIN_PURCHASES`
- `MIN_WEEKS`

**My numbers:** TODO

**Why:** TODO